# Day 4: Pivot Tables and Cross-Tabulation

## Learning Objectives
By the end of this notebook, you will be able to:
- Create pivot tables in Pandas (like Excel)
- Understand index, columns, and values parameters
- Apply multiple aggregation functions
- Add totals (margins) to pivot tables
- Create cross-tabulations
- Know when to use pivot vs groupby

**Duration**: 45 minutes

---

## 1. What is a Pivot Table?

If you've used Excel, you know pivot tables!

**Pivot tables** reshape data to summarize it:
- Rows: Category dimension (e.g., Region)
- Columns: Another dimension (e.g., Product)
- Values: What to aggregate (e.g., Revenue)

Pandas can create Excel-equivalent pivots!

### Demo 1.1: From Long to Wide Format

In [ ]:
# Demo: Data in "Long" Format
# Long format has one row per observation - detailed but hard to read
# This is how most raw data looks (e.g., from databases, CSVs)

import pandas as pd

# "Long" format - each row is a single observation
# Hard to compare regions side-by-side
sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER', 'APAC', 'APAC'],
    'Product': ['Suite A', 'Suite B', 'Suite A', 'Suite B', 'Suite A', 'Suite B'],
    'Revenue': [15000, 8500, 12000, 9200, 11500, 7800]
})

print("Long format (original):")
print(sales)
print("\nThis is hard to read! Let's pivot it...")

### Demo 1.2: Create Your First Pivot Table

In [ ]:
# Demo: Create Your First Pivot Table
# pivot_table() reshapes data from long to wide format (like Excel pivot tables!)
# Syntax: df.pivot_table(index=rows, columns=cols, values=what_to_show, aggfunc=how)

# Create pivot: 
# - index='Region' → Regions become row labels
# - columns='Product' → Products become column headers
# - values='Revenue' → The data to show in cells
# - aggfunc='sum' → How to aggregate (sum, mean, count, etc.)
pivot = sales.pivot_table(
    index='Region',      # What goes in rows
    columns='Product',   # What goes in columns
    values='Revenue',    # What values to show
    aggfunc='sum'        # How to aggregate
)

print("Pivot table (wide format):")
print(pivot)
print("\nMuch easier to read!")

### Exercise 1: Create Basic Pivot

Given sales by quarter and region:
1. Create pivot with Region as rows, Quarter as columns
2. Show total revenue
3. Which region had highest Q1?

In [ ]:
# Exercise 1: Create Basic Pivot Table
# Task: Create a pivot showing revenue by Region and Quarter

# Quarterly regional sales data
quarterly_sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'EMEA', 'EMEA', 'AMER', 'AMER', 'AMER', 'AMER'],
    'Quarter': ['Q1', 'Q2', 'Q3', 'Q4', 'Q1', 'Q2', 'Q3', 'Q4'],
    'Revenue': [125000, 138000, 145000, 152000, 98000, 105000, 112000, 118000]
})

# Create pivot table
# - index: Use 'Region' for rows
# - columns: Use 'Quarter' for columns
# - values: Use 'Revenue' for cell values
# - aggfunc: Use 'sum' to add up values
pivot = quarterly_sales.pivot_table(
    index=___,      # 'Region'
    columns=___,    # 'Quarter'
    values=___,     # 'Revenue'
    aggfunc=___     # 'sum'
)

print("Revenue by Region and Quarter:")
print(pivot)

# Find Q1 leader using idxmax() on the Q1 column
q1_leader = pivot['Q1'].idxmax()
q1_amount = pivot['Q1'].max()
print(f"\nQ1 leader: {q1_leader} with €{q1_amount:,}")

---
## 2. Multiple Aggregations

Show multiple metrics in one pivot table.

### Demo 2.1: Multiple Aggregation Functions

In [ ]:
# Demo: Multiple Aggregation Functions
# You can calculate multiple statistics in one pivot table
# Syntax: aggfunc=['func1', 'func2', 'func3']

# Sample data with multiple transactions per region/product
sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'EMEA', 'AMER', 'AMER', 'AMER'],
    'Product': ['Suite A', 'Suite A', 'Suite B', 'Suite A', 'Suite A', 'Suite B'],
    'Revenue': [15000, 18000, 8500, 12000, 11500, 9200]
})

# Multiple aggregations at once
# Results in multi-level column headers
pivot = sales.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc=['sum', 'mean', 'count']  # Calculate all three!
)

print("Multi-function pivot:")
print(pivot)

### Demo 2.2: Custom Aggregation

In [ ]:
# Demo: Custom Aggregation Functions
# You can define your own aggregation functions
# Pass a dictionary to aggfunc for different functions per column

import numpy as np

# Define a custom function to calculate range (max - min)
def revenue_range(x):
    return x.max() - x.min()

# Apply multiple aggregations including custom function
pivot = sales.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc={
        'Revenue': ['sum', 'mean', revenue_range]  # Mix built-in and custom
    }
)

print("With custom aggregation:")
print(pivot)

---
## 3. Adding Margins (Totals)

Add row and column totals - just like Excel!

### Demo 3.1: Row and Column Totals

In [ ]:
# Demo: Adding Row and Column Totals (Margins)
# margins=True adds totals - just like Excel!
# margins_name sets the label for the total row/column

# Larger dataset for better demonstration
sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'EMEA', 'AMER', 'AMER', 'AMER', 'APAC', 'APAC', 'APAC'],
    'Product': ['Suite A', 'Suite B', 'Analytics', 'Suite A', 'Suite B', 'Analytics', 
                'Suite A', 'Suite B', 'Analytics'],
    'Revenue': [15000, 8500, 11500, 12000, 9200, 10800, 11500, 7800, 9500]
})

# Pivot with totals
pivot = sales.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc='sum',
    margins=True,         # Add row and column totals
    margins_name='Total'  # Label for the totals
)

print("Pivot with row and column totals:")
print(pivot)

# Access the grand total (bottom-right corner)
print("\nGrand total (bottom right): ", pivot.loc['Total', 'Total'])

### Exercise 2: Build Complete Pivot Report

Create a regional product performance report:
1. Pivot by Region (rows) and Product (columns)
2. Show total revenue
3. Add row and column totals
4. Calculate each region's % of total revenue

In [ ]:
# Exercise 2: Build Complete Pivot Report
# Task: Create a regional product performance report with totals and percentages

# Sales data
regional_sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'EMEA', 'AMER', 'AMER', 'AMER', 'APAC', 'APAC', 'APAC'],
    'Product': ['Suite A', 'Suite B', 'Analytics', 'Suite A', 'Suite B', 'Analytics', 
                'Suite A', 'Suite B', 'Analytics'],
    'Revenue': [22500, 15800, 18900, 19200, 14400, 16800, 17500, 12800, 15200]
})

# 1-3. Create pivot with totals
# - index: 'Region'
# - columns: 'Product'
# - values: 'Revenue'
# - aggfunc: 'sum'
# - margins: True (to add totals)
pivot = regional_sales.pivot_table(
    index=___,          # 'Region'
    columns=___,        # 'Product'
    values=___,         # 'Revenue'
    aggfunc=___,        # 'sum'
    margins=___,        # True
    margins_name='Total'
)

print("Regional Product Performance:")
print(pivot)

# 4. Calculate percentages
# Get the grand total from bottom-right corner
grand_total = pivot.loc['Total', 'Total']

# Get regional totals (exclude 'Total' row)
region_totals = pivot.loc[pivot.index != 'Total', 'Total']

# Calculate each region's percentage of total
region_pct = (region_totals / grand_total * 100).round(1)

print("\nRegion % of Total:")
print(region_pct)

---
## 4. Multi-Level Pivots

Multiple dimensions in rows or columns.

### Demo 4.1: Multiple Row Indices

In [ ]:
# Demo: Multi-Level Index (Multiple Row Dimensions)
# You can have multiple levels of grouping in rows
# Syntax: index=['col1', 'col2']

# Data with Region, CustomerType, and Product
sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'EMEA', 'EMEA', 'AMER', 'AMER'],
    'CustomerType': ['Premium', 'Premium', 'Standard', 'Standard', 'Premium', 'Standard'],
    'Product': ['Suite A', 'Suite B', 'Suite A', 'Suite B', 'Suite A', 'Suite B'],
    'Revenue': [15000, 8500, 12000, 7200, 11000, 6800]
})

# Multi-level index: Group by BOTH Region AND CustomerType
pivot = sales.pivot_table(
    index=['Region', 'CustomerType'],  # Two levels of rows!
    columns='Product',
    values='Revenue',
    aggfunc='sum'
)

print("Multi-level pivot:")
print(pivot)

### Demo 4.2: Fill Missing Values

In [ ]:
# Demo: Handling Missing Combinations with fill_value
# Some combinations may not exist in the data - they show as NaN
# fill_value replaces NaN with a specified value (usually 0)

# Data where not all region-product combinations exist
sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'APAC'],  # AMER is missing Suite B!
    'Product': ['Suite A', 'Suite B', 'Suite A', 'Suite A'],
    'Revenue': [15000, 8500, 12000, 11500]
})

# Default behavior: NaN for missing combinations
pivot = sales.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc='sum'
)

print("With NaN for missing combinations:")
print(pivot)

# Better: Fill NaN with 0
pivot_filled = sales.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc='sum',
    fill_value=0  # Replace NaN with 0
)

print("\nWith fill_value=0:")
print(pivot_filled)

---
## 5. Cross-Tabulation

`crosstab()` is similar to pivot but for counts/frequencies.

### Demo 5.1: Basic Cross-Tab

In [ ]:
# Demo: Basic Cross-Tabulation
# crosstab() counts occurrences - perfect for frequency tables
# Syntax: pd.crosstab(index_series, column_series)

# Transaction data - we want to count transactions, not sum values
transactions = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER', 'AMER', 'APAC', 'APAC', 'EMEA'],
    'Product': ['Suite A', 'Suite B', 'Suite A', 'Suite A', 'Suite B', 'Suite A', 'Suite B', 'Suite A']
})

# Count transactions by Region and Product
# This shows HOW MANY transactions, not revenue amounts
crosstab = pd.crosstab(transactions['Region'], transactions['Product'])

print("Transaction counts:")
print(crosstab)
print("\nThis shows how many times each combination occurred.")

### Demo 5.2: Cross-Tab with Percentages

In [ ]:
# Demo: Cross-Tabulation with Margins and Percentages
# margins adds totals, normalize converts counts to percentages

# With margins (totals)
crosstab = pd.crosstab(
    transactions['Region'], 
    transactions['Product'],
    margins=True,           # Add row and column totals
    margins_name='Total'    # Label for totals
)

print("With totals:")
print(crosstab)

# Normalize to percentages
# normalize='index' calculates percentage within each ROW (region)
# normalize='columns' would calculate percentage within each COLUMN
# normalize='all' would calculate percentage of grand total
crosstab_pct = pd.crosstab(
    transactions['Region'], 
    transactions['Product'],
    normalize='index'  # Percentage within each region (row)
) * 100  # Convert to percentage (0-100)

print("\nAs percentages (within region):")
print(crosstab_pct.round(1))

### Exercise 3: Market Share Analysis

Analyze product popularity by region:
1. Create cross-tab of Region × Product
2. Add totals
3. Calculate % of total transactions
4. Find which product is most popular overall

In [ ]:
# Sales transactions
txn_data = pd.DataFrame({
    'Region': ['EMEA']*12 + ['AMER']*10 + ['APAC']*8,
    'Product': (['Suite A']*5 + ['Suite B']*4 + ['Analytics']*3 +  # EMEA
                ['Suite A']*4 + ['Suite B']*3 + ['Analytics']*3 +  # AMER
                ['Suite A']*3 + ['Suite B']*2 + ['Analytics']*3)   # APAC
})

# 1-2. Cross-tab with totals
crosstab = pd.crosstab(
    txn_data['Region'],
    txn_data['Product'],
    margins=True,
    margins_name='Total'
)

print("Transaction counts by Region and Product:")
print(crosstab)

# 3. % of total
grand_total = crosstab.loc['Total', 'Total']
pct_of_total = (crosstab / grand_total * 100).round(1)
print("\n% of Total Transactions:")
print(pct_of_total)

# 4. Most popular product
product_totals = crosstab.loc['Total', crosstab.columns != 'Total']
most_popular = product_totals.idxmax()
print(f"\nMost popular product: {most_popular} ({product_totals.max()} transactions)")

---
## 6. Pivot vs GroupBy: When to Use Each

**Use pivot_table() when:**
- You want cross-tabulated (wide) format
- Creating Excel-style reports
- Comparing categories side-by-side

**Use groupby() when:**
- You want long format results
- Need more complex aggregations
- Chaining multiple operations

### Demo 6.1: Same Analysis, Different Tools

In [ ]:
# Demo: Comparing Pivot vs GroupBy
# Same analysis can be done with either - different output formats
# Pivot = wide format (like Excel), GroupBy = long format

sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER'],
    'Product': ['Suite A', 'Suite B', 'Suite A', 'Suite B'],
    'Revenue': [15000, 8500, 12000, 9200]
})

# GroupBy approach - returns LONG format (Series/DataFrame)
# Good for: Further calculations, chaining operations
grouped = sales.groupby(['Region', 'Product'])['Revenue'].sum()
print("GroupBy result (long):")
print(grouped)

# Pivot approach - returns WIDE format (like spreadsheet)
# Good for: Reports, presentations, easy comparison
pivot = sales.pivot_table(index='Region', columns='Product', values='Revenue', aggfunc='sum')
print("\nPivot result (wide):")
print(pivot)

print("\nSame data, different presentation!")

---
## Challenge Exercise: Executive Dashboard

Create a comprehensive pivot table report:

Given sales data by Region, Product, Quarter:
1. Create pivot: Regions as rows, Products as columns, Revenue as values
2. Add row and column totals
3. Calculate each region's market share (%)
4. Create second pivot: Products as rows, Quarters as columns
5. Identify best product per quarter

In [ ]:
# Annual sales data
annual_sales = pd.DataFrame({
    'Region': ['EMEA']*12 + ['AMER']*12 + ['APAC']*12,
    'Product': ['Suite A', 'Suite B', 'Analytics']*12,
    'Quarter': ['Q1', 'Q1', 'Q1', 'Q2', 'Q2', 'Q2', 'Q3', 'Q3', 'Q3', 'Q4', 'Q4', 'Q4']*3,
    'Revenue': [
        # EMEA
        35000, 22000, 28000, 38000, 24000, 31000, 41000, 26000, 33000, 44000, 28000, 36000,
        # AMER
        28000, 18000, 23000, 31000, 20000, 25000, 34000, 22000, 27000, 37000, 24000, 29000,
        # APAC
        25000, 16000, 21000, 27000, 18000, 23000, 30000, 20000, 25000, 33000, 22000, 27000
    ]
})

# Your solution here
# 1-2. Regional product pivot with totals
regional_pivot = annual_sales.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

print("=" * 60)
print("REGIONAL PRODUCT PERFORMANCE")
print("=" * 60)
print(regional_pivot)

# 3. Market share
grand_total = regional_pivot.loc['Total', 'Total']
market_share = (regional_pivot.loc[regional_pivot.index != 'Total', 'Total'] / grand_total * 100).round(1)
print("\nMarket Share by Region:")
for region, share in market_share.items():
    print(f"  {region}: {share}%")

# 4. Product-quarter pivot
product_pivot = annual_sales.pivot_table(
    index='Product',
    columns='Quarter',
    values='Revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

print("\n" + "=" * 60)
print("PRODUCT PERFORMANCE BY QUARTER")
print("=" * 60)
print(product_pivot)

# 5. Best product per quarter
print("\nBest Product by Quarter:")
for quarter in ['Q1', 'Q2', 'Q3', 'Q4']:
    quarter_data = product_pivot.loc[product_pivot.index != 'Total', quarter]
    best_product = quarter_data.idxmax()
    best_revenue = quarter_data.max()
    print(f"  {quarter}: {best_product} (€{best_revenue:,})")

print("\n" + "=" * 60)

---
## Summary

**You've learned:**
- Creating pivot tables with `pivot_table()`
- Setting index (rows), columns, and values
- Using multiple aggregation functions
- Adding margins (totals)
- Multi-level pivots
- Cross-tabulation with `crosstab()`
- When to use pivot vs groupby

**Key Takeaways:**
- Pivot tables reshape data from long to wide format
- Easier to read than grouped data
- margins=True adds row/column totals
- fill_value=0 replaces NaN in missing combinations
- crosstab() is for counting frequencies
- Pivot tables are perfect for Excel-style reports

**Next:** Joining data from multiple sources!